## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

### Choosing the Correct Embedding Model

Previously, the vector database was created using different embedding models:

-   Hugging Face `all-MiniLM-L6-v2`
-   OpenAI embedding models

For this implementation, the instructor has recreated the database using the original Hugging Face model so everyone can follow along regardless of which model they previously selected.

### Critical rule

> **The embedding model used for retrieval must be the same model used to create the vectors in the database.**

For example:

```
Database vectors:
OpenAI Embeddings
       ↓
Query must also use:
OpenAI Embeddings
```

You cannot do:

```
Database → OpenAI 3072-dimensional vectors

Query → Hugging Face 384-dimensional vector
```

These vectors are incompatible.

This would result in a **dimension mismatch error**.

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [13]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

So what actually happened when we called invoke?It took that fragment."Who is Avery?"It then used the the retrievers encoder that we had set up here.So the embedding function, which is "all-MiniLM-L6-v2" to turn it into a vector, it then called chroma to find the closest vectors. Then located those documents and it's printing them there.

In [14]:
retriever.invoke("Who is Avery?")

[Document(id='999e9acb-b276-436b-ba8b-11184f9a9155', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

We did not passed anything from retriver so LLM doesn't know anything about our documents.

In [15]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery is a given name that can be used for both males and females. It may also refer to various people, characters, or entities depending on the context. Could you please provide more details or specify which Avery you are referring to?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 11, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_8b64c33705', 'id': 'chatcmpl-EKN9cjtMuTHPwONKx0GqeNTBmAXwy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--1cfd3850-dc83-4d3f-90f3-962f1515159f-0', usage_metadata={'input_tokens': 11, 'output_tokens': 48, 'total_tokens': 59, 'input_token_details': {'audio': 0, 'cach

## Time to put this together!

In [16]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [18]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [19]:
answer_question("Who is Averi Lancaster?", [])

'It seems there might be a typo in the name. Based on the information I have, you might be referring to Avery Lancaster. She is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, based in San Francisco, California. Avery has been with the company since 2015 and has played a key role in guiding Insurellm to its current position as a leading Insurance Tech provider. If you meant someone else or need more details, please let me know!'

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

# Building a RAG Pipeline with LangChain — LLM, Retriever, and Temperature

## 1\. Starting Day 3: Building the RAG System

The goal of this session is to build an **expert question-answering system** using **LangChain** and the vector store created previously.

The system will combine two major components:

```
User Question
      ↓
   Retriever
      ↓
Relevant Context
      ↓
      LLM
      ↓
Final Answer
```

The vector store from the previous lesson is reused.

* * *

# 2\. Choosing the Correct Embedding Model

Previously, the vector database was created using different embedding models:

-   Hugging Face `all-MiniLM-L6-v2`
-   OpenAI embedding models

For this implementation, the instructor has recreated the database using the original Hugging Face model so everyone can follow along regardless of which model they previously selected.

### Critical rule

> **The embedding model used for retrieval must be the same model used to create the vectors in the database.**

For example:

```
Database vectors:
OpenAI Embeddings
       ↓
Query must also use:
OpenAI Embeddings
```

You cannot do:

```
Database → OpenAI 3072-dimensional vectors

Query → Hugging Face 384-dimensional vector
```

These vectors are incompatible.

This would result in a **dimension mismatch error**.

* * *

# 3\. LangChain's Model Abstractions

The lesson introduces several LangChain imports, including:

-   `ChatOpenAI`
-   `Chroma`
-   Embedding models
-   Message-related components

One of LangChain's major advantages is that its model abstractions are **swappable**.

For example, you can conceptually replace:

```
ChatOpenAI
```

with:

```
ChatLlama
```

or:

```
ChatAnthropic
```

while still using the common LangChain interface.

The important idea is:

> Different models can follow the same abstraction and respond to operations such as `invoke()`.

This makes it easier to change models without completely rewriting the application.

* * *

# 4\. Creating the Embeddings Object

The first major step is to create the embedding model.

In this lesson:

```
Hugging Face
all-MiniLM-L6-v2
```

is being used.

The embedding model is then associated with the Chroma vector store.

```
Embedding Model
      ↓
all-MiniLM-L6-v2
      ↓
Chroma Vector Store
```

The model must match the model used when the vectors were originally created.

* * *

# 5\. Loading the Existing Chroma Vector Store

The existing vector database is loaded from the `vector_db` directory.

Conceptually:

```
Previously created chunks
        ↓
Embedding Model
        ↓
Vectors
        ↓
Chroma
        ↓
vector_db/
```

The current application reconnects to that stored database using the appropriate embedding model.

* * *

# 6\. Creating the Two Main LangChain Objects

The lesson then creates two major objects:

1.  **Retriever**
2.  **LLM**

These represent the two main parts of the RAG system.

```
                RAG
                 │
       ┌─────────┴─────────┐
       ↓                   ↓
   Retriever              LLM
       ↓                   ↓
Find useful context    Generate answer
```

* * *

# 7\. Creating the Retriever

The Chroma vector store can be converted into a **retriever**.

Conceptually:

```
Chroma Vector Store
        ↓
   as_retriever()
        ↓
    Retriever
```

The benefit is that the retriever provides a simple interface.

You can invoke it with a question:

```
Question
   ↓
Retriever
   ↓
Relevant documents
```

This hides much of the underlying vector-search implementation.

* * *

# 8\. Creating the LLM

The lesson creates a `ChatOpenAI` object for the generative part of the RAG pipeline.

Conceptually:

```
ChatOpenAI
    ↓
Autoregressive LLM
    ↓
Generate final response
```

The model name and **temperature** are supplied when creating the LLM.

The lesson then spends significant time explaining temperature because it is an important parameter for LLM generation.

* * *

# 9\. What Is Temperature?

**Temperature controls the amount of variation in the model's output.**

A lower temperature generally produces more predictable output.

A higher temperature produces more variation in token selection.

```
Temperature = 0
       ↓
More predictable

Temperature ↑
       ↓
More variation
```

It is common to hear temperature described as controlling **"creativity."**

The lecturer considers that description somewhat misleading.

A better way to think about temperature is:

> **It controls how much randomness/variation is introduced when selecting the next token.**

* * *

# 10\. How Temperature Works Internally

An LLM doesn't simply decide:

> "The next word is X."

Instead, it produces probabilities for possible next tokens.

For example:

```
Next token probabilities

"London"    → 60%
"Paris"     → 20%
"airport"   → 10%
"flight"     → 5%
...
```

The system then chooses a token based on these probabilities.

### Temperature 0

The most probable token is effectively selected each time.

```
Highest probability
       ↓
Select it
```

This produces more **predictable and reliable** output.

### Higher temperature

The model becomes more willing to select tokens that aren't the highest-probability choice.

```
Highest probability ──┐
Second choice ────────┤
Third choice ─────────┤ → More variation
...
```

Therefore, higher temperature can make answers more varied—and sometimes stranger.

* * *

# 11\. Temperature ≠ Creativity

The lecturer makes an important distinction:

> **Temperature isn't really a direct creativity control.**

A high temperature can sometimes produce more creative-looking answers, but it fundamentally changes **token selection probabilities**, not creativity itself.

If you specifically want the model to be creative, the recommended approach is to use **prompting**.

For example:

```
System Prompt:
"Respond creatively and use imaginative examples..."
```

You can also explicitly tell the model that you want a response different from previous answers.

So:

```
Want specific behavior?
        ↓
Use prompting/instructions

Want more randomness/variation?
        ↓
Use temperature
```

* * *

# 12\. Temperature of Zero and Reproducibility

A temperature of zero is often described as **deterministic**.

However, the lecturer adds an important qualification:

> **Temperature 0 does not guarantee perfectly reproducible output.**

There can still be randomness and nondeterministic behavior in the overall system.

For stronger reproducibility, a **random seed** can be set.

```
Temperature = 0
       +
Random seed
       ↓
Much more likely to reproduce
the same result
```

Even then, perfect reproducibility is not guaranteed.

* * *

# 13\. Why Perfect Reproducibility Is Difficult

There are several reasons.

### Model changes

Frontier AI providers can update their models.

```
Model Version A
      ↓
Provider updates model
      ↓
Model Version B
```

The same request may therefore behave differently later.

### Parallel computation

Large models perform many calculations in parallel.

Performance optimizations can introduce small differences that prevent exact reproducibility.

Therefore:

> **Nothing is guaranteed to be perfectly reproducible forever.**

The lecturer notes that random seeds will be explored more deeply later in the course.

* * *

# 14\. Why RAG Often Uses Low Temperature

For RAG systems, a temperature of **0** is commonly used because the goal is generally:

-   reliability
-   consistency
-   factual responses
-   less unnecessary variation

The system is retrieving information from a knowledge base, so you generally don't want the model randomly inventing wildly different formulations.

However, other temperatures such as approximately **0.5–0.7** can also be used depending on the application.

* * *

# 15\. The Complete Architecture at This Point

The system now has two key components:

```
                 User Question
                      │
                      ↓
              ┌───────────────┐
              │   Retriever   │
              └───────┬───────┘
                      ↓
              Relevant Documents
                      │
                      ↓
              ┌───────────────┐
              │      LLM      │
              └───────┬───────┘
                      ↓
                  Answer
```

The retriever handles **finding information**.

The LLM handles **generating the response**.

* * *

# 16\. Important Conceptual Separation

This lesson reinforces several distinctions that are easy to confuse:

### Embedding Model

```
Text → Vector
```

### Vector Store

```
Store vectors
+
Search for similar vectors
```

### Retriever

```
Question → Relevant stored documents
```

### LLM

```
Question + Context → Answer
```

Putting them together:

```
Documents
    ↓
Embedding Model
    ↓
Vectors
    ↓
Chroma
    ↓
Retriever
    ↓
Relevant Context
    ↓
LLM
    ↓
Answer
```

* * *

# 17\. Key Practical Lesson

The most important implementation warning in this lesson is:

> **Keep your embedding model consistent between indexing and querying.**

If your database contains vectors generated by one embedding model, use that same model when embedding the user's query.

Otherwise:

```
Different embedding models
        ↓
Different vector spaces
        ↓
Different dimensions/representations
        ↓
Retrieval failure or errors
```

This is one of the most important details when working with vector databases.

* * *

# Key Takeaways

1.  This lesson begins the **actual implementation of a RAG pipeline using LangChain**.
2.  LangChain provides abstractions that make different LLMs relatively **swappable**.
3.  The **embedding model used for queries must match the model used to create the stored vectors**.
4.  Chroma is converted into a **retriever** using `as_retriever()`.
5.  The RAG system has two major components: **retriever + LLM**.
6.  **Temperature controls variation in token selection**, not creativity directly.
7.  Temperature **0** generally gives the most predictable output.
8.  Higher temperature allows lower-probability tokens to be selected, producing more variation.
9.  If you want specific creative behavior, **prompting is generally more reliable than simply increasing temperature**.
10.  Temperature 0 does **not guarantee perfect reproducibility**.
11.  Random seeds can improve reproducibility, but model updates and parallel computation mean perfect reproducibility still isn't guaranteed.
12.  RAG systems commonly use **low temperature** because reliability and consistency are important.
13.  The fundamental RAG flow is:

```
Question
   ↓
Retriever
   ↓
Relevant Context
   ↓
LLM
   ↓
Answer
```

## One-line memory aid

**LangChain connects the retriever to the LLM: use the same embedding model for stored and query vectors, and remember that temperature controls output variation—not creativity itself.**